In [ ]:
from scapy.all import *

target_ip = "172.18.0.2"
broadcast_mac = "ff:ff:ff:ff:ff:ff"
# Defining the interface is necessary for our docker setup + arp
iface = "br-476fde1822dc"  
# Note only not needed when using arping() where scapy sets it automatically

l3 = ARP(pdst=target_ip)
l2 = Ether(dst=broadcast_mac)
# Note scapy will autofill defaults when sending with sr/srp/send

pkt = l2 / l3
pkt

<Ether  dst=ff:ff:ff:ff:ff:ff type=ARP |<ARP  pdst=172.18.0.2 |>>

In [98]:
ans, unans = srp(pkt, iface=iface, verbose=0, timeout=2)
ans, unans

(<Results: TCP:0 UDP:0 ICMP:0 Other:1>,
 <Unanswered: TCP:0 UDP:0 ICMP:0 Other:0>)

In [100]:
ans.summary()

Ether / ARP who has 172.18.0.2 says 172.18.0.1 ==> Ether / ARP is at 02:42:ac:12:00:02 says 172.18.0.2


In [25]:
from scapy.all import *

target_port = 80
tcp_flag = "S"
target_ip = "172.18.0.2"

l4 = TCP(dport=target_port, flags=tcp_flag)
l3 = IP(dst=target_ip)

pkt = l3 / l4
pkt

<IP  frag=0 proto=tcp dst=172.18.0.2 |<TCP  dport=http flags=S |>>

In [26]:
ans, unans = sr(pkt, verbose=0)
ans, unans

(<Results: TCP:1 UDP:0 ICMP:0 Other:0>,
 <Unanswered: TCP:0 UDP:0 ICMP:0 Other:0>)

In [4]:
ans.summary()

IP / TCP 172.18.0.1:ftp_data > 172.18.0.2:http S ==> IP / TCP 172.18.0.2:http > 172.18.0.1:ftp_data RA


In [ ]:
ans[0]  # First tuple (sent, received)

QueryAnswer(query=<IP  frag=0 proto=tcp dst=172.18.0.2 |<TCP  dport=http flags=S |>>, answer=<IP  version=4 ihl=5 tos=0x0 len=40 id=0 flags=DF frag=0 ttl=64 proto=tcp chksum=0xe2a8 src=172.18.0.2 dst=172.18.0.1 |<TCP  sport=http dport=ftp_data seq=0 ack=1 dataofs=5 reserved=0 flags=RA window=0 chksum=0x5744 urgptr=0 |>>)

In [18]:
sent = ans[0].query 
sent

<IP  frag=0 proto=tcp dst=172.18.0.2 |<TCP  dport=http flags=S |>>

In [17]:
sent, received = ans[0]
received

<IP  version=4 ihl=5 tos=0x0 len=40 id=0 flags=DF frag=0 ttl=64 proto=tcp chksum=0xe2a8 src=172.18.0.2 dst=172.18.0.1 |<TCP  sport=http dport=ftp_data seq=0 ack=1 dataofs=5 reserved=0 flags=RA window=0 chksum=0x5744 urgptr=0 |>>

In [105]:
Ether in pkt, ARP in pkt, IP in pkt

(True, True, False)

In [ ]:
res = Ether(pkt.build())
# Equivalently
# res = pkt.__class__(pkt.build())
res

<Ether  dst=ff:ff:ff:ff:ff:ff src=02:42:e7:bd:d5:c4 type=ARP |<ARP  hwtype=Ethernet (10Mb) ptype=IPv4 hwlen=6 plen=4 op=who-has hwsrc=02:42:e7:bd:d5:c4 psrc=172.18.0.1 hwdst=00:00:00:00:00:00 pdst=172.18.0.2 |>>

In [108]:
pkt.getlayer(ARP)

<ARP  pdst=172.18.0.2 |>

In [127]:
pkt.payload

<ARP  pdst=172.18.0.2 |>

In [15]:
for sent, received in ans:

    # Check if the response has a TCP layer
    if received.haslayer(TCP):
        flags = received[TCP].flags
        src_ip = received[IP].src
        sport = received[TCP].sport
        
        # Simple logic to interpret the flags
        status = "Open (SYN-ACK)" if flags == "SA" else "Closed (RST-ACK)"
        
        print(f"{src_ip} | {sport} | {flags} | {status}")

172.18.0.2 | 80 | RA | Closed (RST-ACK)


In [135]:
from scapy.all import *

target_port = 80
target_ip = Net("172.18.0.2", "172.18.0.5")
target_ip = "172.18.0.2"

l4 = UDP(dport=target_port)
l3 = IP(dst=target_ip)

pkt = l3 / l4
pkt

<IP  frag=0 proto=udp dst=172.18.0.2 |<UDP  dport=80 |>>

In [136]:
ans, unans = sr(pkt, verbose=0)
ans, unans

(<Results: TCP:0 UDP:0 ICMP:1 Other:0>,
 <Unanswered: TCP:0 UDP:0 ICMP:0 Other:0>)

reason to send back rst is syn flooding attack